# Preprocessing

Feature engineering for the Uber Dynamic Pricing dataset.

- Encodes categorical columns
- Adds demand/supply ratio + surge flag
- Scales features to [0, 1] with MinMaxScaler
- Saves `X.npy`, `y.npy`, `scaler.pkl` to `data/processed/`

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
RAW_PATH      = os.path.join(PROJECT_ROOT, "data/raw/dynamic_pricing_1M.csv")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data/processed")

print(f"Raw data : {RAW_PATH}")
print(f"Output   : {PROCESSED_DIR}")

Raw data : /home/lux/luxchar/Uber-Dynamic-Pricing/data/raw/dynamic_pricing_1M.csv
Output   : /home/lux/luxchar/Uber-Dynamic-Pricing/data/processed


## Load raw data

In [9]:
df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (1001000, 10)


,Number_of_Riders,Number_of_Drivers,Location_Category,Customer_Loyalty_Status,Number_of_Past_Rides,Average_Ratings,Time_of_Booking,Vehicle_Type,Expected_Ride_Duration,Historical_Cost_of_Ride
0,90,45,Urban,Silver,13,4.47,Night,Premium,90,284.257273
1,58,39,Suburban,Silver,72,4.06,Evening,Economy,43,173.874753
2,42,31,Rural,Silver,0,3.99,Afternoon,Premium,76,329.795469


In [10]:
df.describe()

,Number_of_Riders,Number_of_Drivers,Number_of_Past_Rides,Average_Ratings,Expected_Ride_Duration,Historical_Cost_of_Ride
count,1.001000e+06,1.001000e+06,1.001000e+06,1.001000e+06,1.001000e+06,1.001000e+06
mean,5.997692e+01,2.783450e+01,5.000703e+01,4.256135e+00,9.890773e+01,3.715725e+02
std,2.258350e+01,1.766693e+01,2.701853e+01,4.027678e-01,4.597080e+01,1.738087e+02
min,1.000000e+00,1.000000e+00,0.000000e+00,3.500000e+00,1.000000e+00,2.599345e+01
25%,4.400000e+01,1.400000e+01,3.000000e+01,3.960000e+00,6.600000e+01,2.464677e+02
50%,6.000000e+01,2.700000e+01,5.000000e+01,4.260000e+00,1.000000e+02,3.725187e+02
75%,7.600000e+01,4.000000e+01,7.000000e+01,4.550000e+00,1.330000e+02,4.984773e+02
max,1.000000e+02,8.900000e+01,1.000000e+02,5.000000e+00,1.800000e+02,8.361164e+02


## Encode categoricals

In [11]:
LOCATION_MAP = {"Urban": 0, "Suburban": 1, "Rural": 2}
LOYALTY_MAP  = {"Regular": 0, "Silver": 1, "Gold": 2}
TIME_MAP     = {"Morning": 0, "Afternoon": 1, "Evening": 2, "Night": 3}
VEHICLE_MAP  = {"Economy": 0, "Premium": 1}

df["Location_Category"]      = df["Location_Category"].map(LOCATION_MAP)
df["Customer_Loyalty_Status"] = df["Customer_Loyalty_Status"].map(LOYALTY_MAP)
df["Time_of_Booking"]         = df["Time_of_Booking"].map(TIME_MAP)
df["Vehicle_Type"]            = df["Vehicle_Type"].map(VEHICLE_MAP)

df.head(3)

,Number_of_Riders,Number_of_Drivers,Location_Category,Customer_Loyalty_Status,Number_of_Past_Rides,Average_Ratings,Time_of_Booking,Vehicle_Type,Expected_Ride_Duration,Historical_Cost_of_Ride
0,90,45,0,1,13,4.47,3,1,90,284.257273
1,58,39,1,1,72,4.06,2,0,43,173.874753
2,42,31,2,1,0,3.99,1,1,76,329.795469


## Feature engineering

In [12]:
df["demand_supply_ratio"] = df["Number_of_Riders"] / (df["Number_of_Drivers"] + 1e-6)
df["is_surge"] = (df["demand_supply_ratio"] > df["demand_supply_ratio"].median()).astype(int)

print(f"Surge rate: {df['is_surge'].mean():.1%}")
print(f"DSR range : [{df['demand_supply_ratio'].min():.2f}, {df['demand_supply_ratio'].max():.2f}]")

Surge rate: 50.0%
DSR range : [0.01, 100.00]


## Scale & save

In [13]:
STATE_COLS = [
    "Number_of_Riders",
    "Number_of_Drivers",
    "demand_supply_ratio",
    "is_surge",
    "Location_Category",
    "Customer_Loyalty_Status",
    "Number_of_Past_Rides",
    "Average_Ratings",
    "Time_of_Booking",
    "Vehicle_Type",
    "Expected_Ride_Duration",
]

scaler = MinMaxScaler()
X = scaler.fit_transform(df[STATE_COLS])
y = df["Historical_Cost_of_Ride"].values

print(f"X shape : {X.shape}")
print(f"y range : [{y.min():.1f}, {y.max():.1f}]")
print(f"Features: {STATE_COLS}")

X shape : (1001000, 11)
y range : [26.0, 836.1]
Features: ['Number_of_Riders', 'Number_of_Drivers', 'demand_supply_ratio', 'is_surge', 'Location_Category', 'Customer_Loyalty_Status', 'Number_of_Past_Rides', 'Average_Ratings', 'Time_of_Booking', 'Vehicle_Type', 'Expected_Ride_Duration']


In [14]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
np.save(os.path.join(PROCESSED_DIR, "X.npy"), X)
np.save(os.path.join(PROCESSED_DIR, "y.npy"), y)
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "scaler.pkl"))
print(f"Saved to {PROCESSED_DIR}")

Saved to /home/lux/luxchar/Uber-Dynamic-Pricing/data/processed
